# Notebook 01 — ETL Pipeline — Student Version

## Goal

Build a **versioned, privacy-aware ML feature dataset** from the QBC12 Airbnb PostgreSQL database.

Final output:

- one row per `listing_id`
- one fixed `cutoff_date`
- features built only from data available before/on the cutoff
- target built from future calendar availability
- no raw PII columns in the final ML dataset

The next notebook will use this output for MLflow experiments. If this ETL is messy, the ML notebook will be garbage.

## What you must submit from this notebook

By the end, your notebook must save these files under `data/features/`:

```text
listing_availability_features_<version>.csv
listing_availability_features_<version>.parquet
listing_availability_features_<version>_metadata.json
listing_availability_features_<version>_validation_report.json
pii_audit_<version>.csv
```

The notebook must also show:

1. database connection check,
2. table/column inspection,
3. PII audit,
4. cutoff-date logic,
5. feature construction,
6. label construction,
7. validation checks.

## 0. Imports

These libraries are enough for the ETL notebook.

Install missing packages with:

```bash
pip install pandas numpy sqlalchemy psycopg2-binary pyarrow
```

In [123]:
! pip install pandas numpy sqlalchemy psycopg2-binary pyarrow

Looking in indexes: https://package-mirror.liara.ir/repository/pypi/simple


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [124]:
import os
import json
import re
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv

load_dotenv(".env")

True

## 1. Configuration

These values define the dataset version and the time windows.

- `PAST_WINDOW_DAYS`: how much history is used for features.
- `FUTURE_WINDOW_DAYS`: how much future data is used for the target.
- `HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD`: the rule for the positive class.

If you change any of these, change `DATASET_VERSION`.

In [125]:
# -----------------------------
# ETL Configuration
# -----------------------------
DATASET_VERSION = "v1_student"

ENTITY_COLUMN = "listing_id"

PAST_WINDOW_DAYS = 90
FUTURE_WINDOW_DAYS = 30
HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD = 0.30

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
FEATURE_DIR = DATA_DIR / "features"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURE_DIR:", FEATURE_DIR)

PROJECT_ROOT: c:\Users\aidah\Documents\MLOPS\HW_A
FEATURE_DIR: c:\Users\aidah\Documents\MLOPS\HW_A\data\features


## 2. Database connection

Use your assigned student database user.

The QBC12 database is:

host: 185.50.38.163

port: 32112

database: qbc12_airbnb

Important:

- Keep `sslmode=disable`.
- Do not commit real passwords to Git.

In [139]:
# -----------------------------
# Database Connection
# -----------------------------

# Clear old environment variables that may point to the wrong database.
# for key in ["PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]:
#     os.environ.pop(key, None)

DB_HOST = os.getenv("PGHOST", "185.50.38.163")
DB_PORT_RAW = os.getenv("PGPORT", "32112")
DB_NAME = os.getenv("PGDATABASE", "qbc12_airbnb")
DB_USER = os.getenv("PGUSER", "")
DB_PASSWORD = os.getenv("PGPASSWORD", "")

if not DB_USER or not DB_PASSWORD:
    raise ValueError(
        "Please set PGUSER and PGPASSWORD with your assigned database credentials."
    )

try:
    DB_PORT = int(DB_PORT_RAW)
except ValueError:
    raise ValueError(f"Invalid PGPORT value: {DB_PORT_RAW!r}. It must be an integer.")

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(db_url)

def read_sql(query, params=None):
    if params is None:
        return pd.read_sql_query(query, con=engine)
    return pd.read_sql_query(text(query), con=engine, params=params)

# quick connection check
conn_check = read_sql("""
SELECT
    current_database() AS current_database,
    current_user AS current_user,
    inet_server_addr() AS server_addr,
    inet_server_port() AS server_port,
    now() AS server_time;
""")

display(conn_check)

,current_database,current_user,server_addr,server_port,server_time
0,qbc12_airbnb,student_ayda_hafezian,172.19.0.2,5432,2026-06-02 19:30:04.449334+00:00


## 3. Inspect the available data

Before writing ETL, inspect the database.

You should confirm:

- which tables exist,
- which columns exist,
- how many rows each table has,
- whether important fields are missing.

In [140]:
tables_df = read_sql("""
SELECT
    table_schema,
    table_name,
    table_type
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
""")

tables_df

,table_schema,table_name,table_type
0,core,calendar_day,BASE TABLE
1,core,host,BASE TABLE
2,core,listing,BASE TABLE
3,core,neighbourhood,BASE TABLE
4,core,review,BASE TABLE


In [141]:
columns_df = read_sql("""
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'core'
ORDER BY table_schema, table_name, ordinal_position;
""")

columns_df

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable
0,core,calendar_day,1,listing_id,bigint,NO
1,core,calendar_day,2,date,date,NO
2,core,calendar_day,3,available,boolean,YES
3,core,calendar_day,4,price,numeric,YES
4,core,calendar_day,5,adjusted_price,numeric,YES
5,core,calendar_day,6,minimum_nights,integer,YES
6,core,calendar_day,7,maximum_nights,integer,YES
7,core,host,1,host_id,bigint,NO
8,core,host,2,host_pseudo_id,text,NO
9,core,host,3,is_superhost,boolean,YES


In [142]:
row_counts_df = read_sql("""
SELECT 'core.calendar_day' AS table_name, COUNT(*) AS row_count FROM core.calendar_day
UNION ALL
SELECT 'core.host' AS table_name, COUNT(*) AS row_count FROM core.host
UNION ALL
SELECT 'core.listing' AS table_name, COUNT(*) AS row_count FROM core.listing
UNION ALL
SELECT 'core.neighbourhood' AS table_name, COUNT(*) AS row_count FROM core.neighbourhood
UNION ALL
SELECT 'core.review' AS table_name, COUNT(*) AS row_count FROM core.review
ORDER BY table_name;
""")

row_counts_df

,table_name,row_count
0,core.calendar_day,3825200
1,core.host,9201
2,core.listing,10480
3,core.neighbourhood,22
4,core.review,501084


## 4. Data quality audit

This step decides which columns are safe and useful.

You must check at least:

1. calendar date range,
2. review date range,
3. whether `calendar_day.price` and `adjusted_price` are usable,
4. whether recent review windows are meaningful.

Do not include columns that are all-null or nearly useless.

In [143]:
calendar_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE price IS NULL) AS null_price,
    COUNT(*) FILTER (WHERE adjusted_price IS NULL) AS null_adjusted_price,
    COUNT(*) FILTER (WHERE available IS NULL) AS null_available,
    MIN(date) AS min_calendar_date,
    MAX(date) AS max_calendar_date
FROM core.calendar_day;
""")

review_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE comment_len IS NULL) AS null_comment_len,
    MIN(review_date) AS min_review_date,
    MAX(review_date) AS max_review_date
FROM core.review;
""")

display(calendar_quality_df)
display(review_quality_df)

,n_rows,null_price,null_adjusted_price,null_available,min_calendar_date,max_calendar_date
0,3825200,3825200,3825200,0,2025-09-11,2026-09-10


,n_rows,null_comment_len,min_review_date,max_review_date
0,501084,0,2010-08-22,2025-09-11


In [144]:
# Inspect small samples.
# Keep LIMIT small. Do not pull full raw calendar/review tables into Pandas.

for table_name in ["listing", "host", "neighbourhood", "review", "calendar_day"]:
    print(f"\n===== core.{table_name} =====")
    display(read_sql(f"SELECT * FROM core.{table_name} LIMIT 10;"))


===== core.listing =====


,listing_id,host_id,neighbourhood_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms_text,listing_price,minimum_nights,maximum_nights,instant_bookable,license
0,27886,97647,2,Private room,Private room in houseboat,2,1.0,1.0,1.5 baths,132.0,3,356,False,0363 974D 4986 7411 88D8
1,28871,124245,2,Private room,Private room in rental unit,2,1.0,1.0,1 shared bath,89.0,2,730,False,0363 607B EA74 0BD8 2F6F
2,29051,124245,1,Private room,Private room in condo,2,1.0,1.0,1 shared bath,61.0,2,730,False,0363 607B EA74 0BD8 2F6F
3,44391,194779,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,730,False,0363 E76E F06A C1DD 172C
4,48373,220434,8,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,1125,False,0363 4A2B A6AD 0196 F684
5,49552,225987,2,Entire home/apt,Entire guest suite,3,2.0,2.0,1 bath,322.0,3,1125,False,0363 576A D827 5085 6B83
6,50263,230246,1,Entire home/apt,Entire condo,4,2.0,3.0,1.5 baths,457.0,2,14,False,0363 7F3D 0BAE 28C8 C7D2
7,50515,231864,14,Entire home/apt,Entire townhouse,5,3.0,3.0,1.5 baths,198.0,7,18,False,0363 5DDB E495 A6D5 CEC6
8,50523,231946,2,Entire home/apt,Entire condo,2,1.0,1.0,1 bath,162.0,2,365,False,0363 22DC 0E52 B70B 0FB8
9,53921,252245,10,Entire home/apt,Entire rental unit,3,1.0,NaN,1 bath,NaN,1,21,False,0363 B43C B1D4 2666 3739



===== core.host =====


,host_id,host_pseudo_id,is_superhost
0,27837566,12a252de05fbf2f7ba9f57fa3baa099acd17e2a9c7efc7...,False
1,12840373,d9f7e79668b99a5cb7963bfb2430d8b6b960a0ec4e82bb...,False
2,226859324,cc90d30412e286c0525c7914d031807c8c00e017fe1915...,True
3,20204265,755d79bf4be51df2d85792123200e6641db8589551965e...,True
4,47981094,b40c45156e81696746b6eb51ef86aeccff7c6d7cd5eac2...,False
5,443406859,cfe69dd56cffef559069b30216f8954b57e1de886b92a3...,False
6,2626085,43c4cdd7f1413364dd809c6c92c20baed35faa51f84e76...,False
7,74999205,1033d2369452b0969c1f63061af401f4581a92873b5659...,True
8,7969106,1b41831025e74b04a73fe88e99233d09f39660eac85bf6...,False
9,6127483,8cffc835ea5e24b102cf446e9d369edad9f3a2bf91bfbc...,True



===== core.neighbourhood =====


,neighbourhood_id,name
0,1,Centrum-Oost
1,2,Centrum-West
2,3,Oostelijk Havengebied - Indische Buurt
3,4,Westerpark
4,5,Slotervaart
5,6,Bijlmer-Centrum
6,7,Geuzenveld - Slotermeer
7,8,Buitenveldert - Zuidas
8,9,Noord-West
9,10,IJburg - Zeeburgereiland



===== core.review =====


,review_id,listing_id,review_date,reviewer_id,reviewer_pseudo_id,comment_len
0,531281017,18062995,2019-09-17,73925523,2408ff6b678c0bdc30857dc6631d8762b57159ef9741c3...,392
1,531788922,18062995,2019-09-18,82572265,fb2e9dd6e8225de8231d388c9b9cf92b9c88f7e0823389...,13
2,532695264,18062995,2019-09-20,100595030,2503cafbac72a2a0a211282e2ed6e97c60058769ab3cf8...,467
3,537292491,18062995,2019-09-28,282542584,2612adccbea98416289b033feb314e33a3947244104fcf...,78
4,540324583,18062995,2019-10-03,116115856,7f6c33a5339d594d253c1aa0c9d5a2ae714321907af14c...,220
5,540880279,18062995,2019-10-04,65938182,b280baa9d72baf439c77b54c6ee11a42e4a9403fccec35...,145
6,542086221,18062995,2019-10-06,155392872,964cb3a74d1ccaaadd95915b842090390eed7754785aa5...,270
7,543734173,18062995,2019-10-08,52878420,5f477eaea2d8cd299828be54f31613eabe9e17072ff0bc...,204
8,544447121,18062995,2019-10-10,120972473,1314f5367fdcd67e16c7fb0828dad3be64be054d8042c2...,92
9,547826338,18062995,2019-10-16,31002341,6eedc8878c30f4380cbc920fd5ed63ff122aeba8f3bdf6...,91



===== core.calendar_day =====


,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,857771032141263073,2026-06-20,False,None,None,2,45
1,857771032141263073,2026-06-21,False,None,None,2,45
2,857771032141263073,2026-06-22,False,None,None,2,45
3,857771032141263073,2026-06-23,False,None,None,2,45
4,857771032141263073,2026-06-24,False,None,None,2,45
5,857771032141263073,2026-06-25,False,None,None,2,45
6,857771032141263073,2026-06-26,False,None,None,2,45
7,857771032141263073,2026-06-27,False,None,None,2,45
8,857771032141263073,2026-06-28,False,None,None,2,45
9,857771032141263073,2026-06-29,False,None,None,2,45


## 5. Choose the cutoff date

The cutoff separates features from the label.

Rules:

- Historical features use dates `history_start_date <= date <= cutoff_date`.
- The label uses dates `cutoff_date < date <= label_end_date`.
- The cutoff must have enough past calendar data and enough future calendar data.

For this homework, use a 90-day feature window and a 30-day label window.

In [145]:
range_df = read_sql("""
SELECT
    (SELECT MIN(date) FROM core.calendar_day) AS calendar_min_date,
    (SELECT MAX(date) FROM core.calendar_day) AS calendar_max_date,
    (SELECT MIN(review_date) FROM core.review) AS review_min_date,
    (SELECT MAX(review_date) FROM core.review) AS review_max_date;
""")

range_df

,calendar_min_date,calendar_max_date,review_min_date,review_max_date
0,2025-09-11,2026-09-10,2010-08-22,2025-09-11


In [146]:
calendar_min_date = pd.to_datetime(range_df.loc[0, "calendar_min_date"]).date()
calendar_max_date = pd.to_datetime(range_df.loc[0, "calendar_max_date"]).date()

earliest_cutoff_allowed_by_calendar = calendar_min_date + timedelta(days=PAST_WINDOW_DAYS)
latest_cutoff_allowed_by_calendar = calendar_max_date - timedelta(days=FUTURE_WINDOW_DAYS)

cutoff_date = latest_cutoff_allowed_by_calendar
history_start_date = cutoff_date - timedelta(days=PAST_WINDOW_DAYS)
label_end_date = cutoff_date + timedelta(days=FUTURE_WINDOW_DAYS)

print("calendar_min_date:", calendar_min_date)
print("calendar_max_date:", calendar_max_date)
print("earliest_cutoff_allowed_by_calendar:", earliest_cutoff_allowed_by_calendar)
print("latest_cutoff_allowed_by_calendar:", latest_cutoff_allowed_by_calendar)
print("cutoff_date:", cutoff_date)
print("history_start_date:", history_start_date)
print("label_end_date:", label_end_date)

assert earliest_cutoff_allowed_by_calendar <= cutoff_date <= latest_cutoff_allowed_by_calendar
assert history_start_date >= calendar_min_date
assert label_end_date <= calendar_max_date
assert history_start_date <= cutoff_date < label_end_date

calendar_min_date: 2025-09-11
calendar_max_date: 2026-09-10
earliest_cutoff_allowed_by_calendar: 2025-12-10
latest_cutoff_allowed_by_calendar: 2026-08-11
cutoff_date: 2026-08-11
history_start_date: 2026-05-13
label_end_date: 2026-09-10


## 6. PII audit

Raw identifiers can be needed for joins, but they must not become model features.

Your final ML feature table must not contain:

- `host_id`
- `host_pseudo_id`
- `review_id`
- `reviewer_id`
- `reviewer_pseudo_id`
- `license`
- raw text fields that may contain sensitive information

`listing_id` may stay as an entity key, but it must be excluded from model inputs later.

In [147]:
# TODO: complete the PII audit table.
# Add rows for all sensitive or identity-linking columns you find relevant.

pii_audit = pd.DataFrame([
    {
        "table": "listing",
        "column": "listing_id",
        "pii_type": "entity identifier",
        "decision": "keep as entity key only",
        "reason": "needed to define one row per listing; not a model input"
    },
    {
        "table": "listing",
        "column": "host_id",
        "pii_type": "direct identifier / join key",
        "decision": "drop from final dataset",
        "reason": "used only for joins and host-level aggregation"
    },
    {
        "table": "host",
        "column": "host_pseudo_id",
        "pii_type": "pseudonymous identifier",
        "decision": "drop from final dataset",
        "reason": "identity-linking column; not needed as a model feature"
    },
    {
        "table": "listing",
        "column": "license",
        "pii_type": "regulated / sensitive identifier",
        "decision": "drop from final dataset",
        "reason": "may expose regulated registration details"
    },
    {
        "table": "review",
        "column": "review_id",
        "pii_type": "record identifier",
        "decision": "drop from final dataset",
        "reason": "not useful as an ML feature"
    },
    {
        "table": "review",
        "column": "reviewer_id",
        "pii_type": "user identifier",
        "decision": "drop from final dataset",
        "reason": "identity-linking and not needed in final feature table"
    },
    {
        "table": "review",
        "column": "reviewer_pseudo_id",
        "pii_type": "pseudonymous user identifier",
        "decision": "drop from final dataset",
        "reason": "identity-linking and not needed in final feature table"
    },
    {
        "table": "review",
        "column": "comments",
        "pii_type": "raw free text",
        "decision": "drop from final dataset",
        "reason": "raw text may contain personal or sensitive information"
    },
    {
        "table": "listing",
        "column": "bathrooms_text",
        "pii_type": "raw text",
        "decision": "transform then drop raw column",
        "reason": "convert to numeric bathrooms feature, do not keep raw text"
    },
])

pii_audit

,table,column,pii_type,decision,reason
0,listing,listing_id,entity identifier,keep as entity key only,needed to define one row per listing; not a mo...
1,listing,host_id,direct identifier / join key,drop from final dataset,used only for joins and host-level aggregation
2,host,host_pseudo_id,pseudonymous identifier,drop from final dataset,identity-linking column; not needed as a model...
3,listing,license,regulated / sensitive identifier,drop from final dataset,may expose regulated registration details
4,review,review_id,record identifier,drop from final dataset,not useful as an ML feature
5,review,reviewer_id,user identifier,drop from final dataset,identity-linking and not needed in final featu...
6,review,reviewer_pseudo_id,pseudonymous user identifier,drop from final dataset,identity-linking and not needed in final featu...
7,review,comments,raw free text,drop from final dataset,raw text may contain personal or sensitive inf...
8,listing,bathrooms_text,raw text,transform then drop raw column,"convert to numeric bathrooms feature, do not k..."


## 7. Extract static tables

`listing`, `host`, and `neighbourhood` are small enough to load directly.

Do not load full `review` or `calendar_day` into Pandas. Those must be aggregated in SQL later.

In [148]:
# TODO: write SQL to load the required listing columns.
listing_df = read_sql("""
SELECT
    listing_id,
    host_id,
    neighbourhood_id,
    room_type,
    property_type,
    accommodates,
    bedrooms,
    beds,
    bathrooms_text,
    minimum_nights,
    maximum_nights,
    instant_bookable,
    license
FROM core.listing;
""")

# TODO: write SQL to load host columns.
host_df = read_sql("""
SELECT
    host_id,
    host_pseudo_id,
    is_superhost
FROM core.host;
""")

# TODO: write SQL to load neighbourhood columns.
neighbourhood_df = read_sql("""
SELECT
    neighbourhood_id,
    name AS neighbourhood_name
FROM core.neighbourhood;
""")

print("listing:", listing_df.shape)
print("host:", host_df.shape)
print("neighbourhood:", neighbourhood_df.shape)

listing: (10480, 13)
host: (9201, 3)
neighbourhood: (22, 2)


## 8. Clean static fields

Convert database values into ML-friendly columns.

Required work:

- convert booleans to boolean dtype,
- convert numeric listing columns to numeric dtype,
- parse `bathrooms_text` into a numeric `bathrooms` feature.

In [149]:
# TODO: normalize boolean columns.
# Example target columns:
# - listing_df["instant_bookable"]
# - host_df["is_superhost"]

def normalize_boolean(series: pd.Series) -> pd.Series:
    true_values = {"t", "true", "1", "yes", "y"}
    false_values = {"f", "false", "0", "no", "n"}

    def convert(x):
        if pd.isna(x):
            return pd.NA
        if isinstance(x, bool):
            return x

        value = str(x).strip().lower()
        if value in true_values:
            return True
        if value in false_values:
            return False
        return pd.NA

    return series.apply(convert).astype("boolean")


listing_df["instant_bookable"] = normalize_boolean(listing_df["instant_bookable"])
host_df["is_superhost"] = normalize_boolean(host_df["is_superhost"])


# TODO: normalize numeric listing columns.

# Make sure expected numeric columns exist
expected_numeric_cols = [
    "accommodates",
    "bedrooms",
    "beds",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
]

for col in expected_numeric_cols:
    if col not in listing_df.columns:
        listing_df[col] = np.nan

# Clean listing_price only if present
listing_df["listing_price"] = (
    listing_df["listing_price"]
    .replace({None: np.nan})
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
    .str.replace(r"[^\d.\-]", "", regex=True)
)

for col in expected_numeric_cols:
    listing_df[col] = pd.to_numeric(listing_df[col], errors="coerce")

def parse_bathrooms(text_value):
    """
    Convert bathrooms_text into a number.

    Examples:
    - '1 bath' -> 1.0
    - '1.5 baths' -> 1.5
    - 'Half-bath' -> 0.5
    - missing/unrecognized -> NaN
    """
    if pd.isna(text_value):
        return np.nan

    text_value = str(text_value).strip().lower()

    if text_value in {"", "nan", "none"}:
        return np.nan

    if "half-bath" in text_value or "half bath" in text_value:
        return 0.5

    match = re.search(r"(\d+(\.\d+)?)", text_value)
    if match:
        return float(match.group(1))

    return np.nan


listing_df["bathrooms"] = listing_df["bathrooms_text"].apply(parse_bathrooms)

listing_df[[
    "bathrooms_text",
    "bathrooms",
    "listing_price",
    "accommodates",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights"
]].head(10)

,bathrooms_text,bathrooms,listing_price,accommodates,bedrooms,beds,minimum_nights,maximum_nights
0,1.5 baths,1.5,<NA>,2,1.0,1.0,3,356
1,1 shared bath,1.0,<NA>,2,1.0,1.0,2,730
2,1 shared bath,1.0,<NA>,2,1.0,1.0,2,730
3,1.5 baths,1.5,<NA>,4,2.0,NaN,3,730
4,1.5 baths,1.5,<NA>,4,2.0,NaN,3,1125
5,1 bath,1.0,<NA>,3,2.0,2.0,3,1125
6,1.5 baths,1.5,<NA>,4,2.0,3.0,2,14
7,1.5 baths,1.5,<NA>,5,3.0,3.0,7,18
8,1 bath,1.0,<NA>,2,1.0,1.0,2,365
9,1 bath,1.0,<NA>,3,1.0,NaN,1,21


## 9. Build static listing features

Join:

- `listing` → `host`
- `listing` → `neighbourhood`
- host-level aggregate `host_listing_count`

Final static features should be one row per `listing_id`.

Do not keep raw `host_id`, `host_pseudo_id`, `neighbourhood_id`, `license`, or `bathrooms_text` in the final static feature table.

In [150]:
# TODO: create host_listing_features with one row per host_id.
host_listing_features = (
    listing_df.groupby("host_id", dropna=False)
    .agg(host_listing_count=("listing_id", "nunique"))
    .reset_index()
)


# TODO: merge listing, host, host_listing_features, and neighbourhood.
base_listing_features = (
    listing_df
    .merge(host_df, on="host_id", how="left")
    .merge(host_listing_features, on="host_id", how="left")
    .merge(neighbourhood_df, on="neighbourhood_id", how="left")
)


# TODO: choose privacy-safe static feature columns.
static_feature_cols = [
    "listing_id",
    "room_type",
    "property_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable",
    "is_superhost",
    "host_listing_count",
    "neighbourhood_name",
]

static_features = base_listing_features[static_feature_cols].copy()

assert static_features["listing_id"].duplicated().sum() == 0

print(static_features.shape)
static_features.head()


static_features = base_listing_features[static_feature_cols].copy()

assert static_features["listing_id"].duplicated().sum() == 0

print(static_features.shape)
static_features.head()

(10480, 14)
(10480, 14)


,listing_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,instant_bookable,is_superhost,host_listing_count,neighbourhood_name
0,27886,Private room,Private room in houseboat,2,1.0,1.0,1.5,<NA>,3,356,False,True,1,Centrum-West
1,28871,Private room,Private room in rental unit,2,1.0,1.0,1.0,<NA>,2,730,False,True,2,Centrum-West
2,29051,Private room,Private room in condo,2,1.0,1.0,1.0,<NA>,2,730,False,True,2,Centrum-Oost
3,44391,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,<NA>,3,730,False,False,1,Centrum-Oost
4,48373,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,<NA>,3,1125,False,False,1,Buitenveldert - Zuidas


## 10. Build review features in SQL

Do not load raw `core.review` into Pandas.

Build one row per listing in SQL.

Required output columns:

- `listing_id`
- `total_reviews_before_cutoff`
- `unique_reviewers_before_cutoff`
- `avg_comment_len_before_cutoff`
- `max_comment_len_before_cutoff`
- `days_since_last_review`

Use only reviews where `review_date <= cutoff_date`.

In [151]:
# TODO: write SQL aggregation over core.review.
# Use CAST(:cutoff_date AS date) when using the cutoff inside SQL.

review_features = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*) AS total_reviews_before_cutoff,
        COUNT(DISTINCT reviewer_id) AS unique_reviewers_before_cutoff,
        AVG(comment_len) AS avg_comment_len_before_cutoff,
        MAX(comment_len) AS max_comment_len_before_cutoff,
        CAST(CAST(:cutoff_date AS date) - MAX(review_date) AS integer) AS days_since_last_review
    FROM core.review
    WHERE review_date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={"cutoff_date": cutoff_date},
)


# TODO: convert feature columns to numeric where needed.

review_numeric_cols = [
    "total_reviews_before_cutoff",
    "unique_reviewers_before_cutoff",
    "avg_comment_len_before_cutoff",
    "max_comment_len_before_cutoff",
    "days_since_last_review",
]

for col in review_numeric_cols:
    review_features[col] = pd.to_numeric(review_features[col], errors="coerce")

assert review_features["listing_id"].duplicated().sum() == 0

print(review_features.shape)
review_features.head()

(9383, 6)


,listing_id,total_reviews_before_cutoff,unique_reviewers_before_cutoff,avg_comment_len_before_cutoff,max_comment_len_before_cutoff,days_since_last_review
0,27886,311,311,302.167203,1917,338
1,28871,732,729,201.236339,1265,338
2,29051,849,841,245.108363,2253,337
3,44391,42,42,242.309524,891,1452
4,48373,5,5,272.200000,949,835


## 11. Build calendar history features in SQL

Do not load raw `core.calendar_day` into Pandas.

Build historical availability features using:

- 90-day history window,
- 30-day recent history window.

Do not include calendar price features unless your audit proves they are usable.

In [152]:
# TODO: write SQL aggregation over core.calendar_day.
# Use history_start_date and cutoff_date.
#
# Required output examples:
# - available_days_last_90d
# - available_rate_last_90d
# - avg_minimum_nights_calendar_last_90d
# - avg_maximum_nights_calendar_last_90d
# - available_days_last_30d
# - available_rate_last_30d
# - avg_minimum_nights_calendar_last_30d
# - avg_maximum_nights_calendar_last_30d

calendar_features_all = read_sql(
    """
    SELECT
        listing_id,

        COUNT(*) FILTER (
            WHERE available = TRUE
        ) AS available_days_last_90d,

        AVG(
            CASE WHEN available = TRUE THEN 1.0 ELSE 0.0 END
        ) AS available_rate_last_90d,

        AVG(minimum_nights) AS avg_minimum_nights_calendar_last_90d,
        AVG(maximum_nights) AS avg_maximum_nights_calendar_last_90d,

        COUNT(*) FILTER (
            WHERE date > CAST(:cutoff_date AS date) - INTERVAL '30 day'
              AND available = TRUE
        ) AS available_days_last_30d,

        AVG(
            CASE
                WHEN date > CAST(:cutoff_date AS date) - INTERVAL '30 day'
                THEN CASE WHEN available = TRUE THEN 1.0 ELSE 0.0 END
            END
        ) AS available_rate_last_30d,

        AVG(
            CASE
                WHEN date > CAST(:cutoff_date AS date) - INTERVAL '30 day'
                THEN minimum_nights
            END
        ) AS avg_minimum_nights_calendar_last_30d,

        AVG(
            CASE
                WHEN date > CAST(:cutoff_date AS date) - INTERVAL '30 day'
                THEN maximum_nights
            END
        ) AS avg_maximum_nights_calendar_last_30d

    FROM core.calendar_day
    WHERE date >= CAST(:history_start_date AS date)
      AND date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "history_start_date": history_start_date,
        "cutoff_date": cutoff_date,
    },
)

# TODO: convert numeric columns.

calendar_numeric_cols = [
    "available_days_last_90d",
    "available_rate_last_90d",
    "avg_minimum_nights_calendar_last_90d",
    "avg_maximum_nights_calendar_last_90d",
    "available_days_last_30d",
    "available_rate_last_30d",
    "avg_minimum_nights_calendar_last_30d",
    "avg_maximum_nights_calendar_last_30d",
]

for col in calendar_numeric_cols:
    calendar_features_all[col] = pd.to_numeric(calendar_features_all[col], errors="coerce")

assert calendar_features_all["listing_id"].duplicated().sum() == 0

print(calendar_features_all.shape)
calendar_features_all.head()

(10480, 9)


,listing_id,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d
0,27886,0,0.000000,3.0,30.0,0,0.000000,3.0,30.0
1,28871,46,0.505495,2.0,730.0,14,0.466667,2.0,730.0
2,29051,43,0.472527,2.0,730.0,16,0.533333,2.0,730.0
3,44391,0,0.000000,3.0,730.0,0,0.000000,3.0,730.0
4,48373,0,0.000000,3.0,1125.0,0,0.000000,3.0,1125.0


## 12. Build the target label

The label is built from future calendar availability.

Positive class:

```text
high_demand_proxy = 1 if future_available_rate_30d <= 0.30
```

This is not confirmed booking demand. It is a low-availability proxy.

In [153]:
# TODO: write SQL to build one label row per listing.
# Use only dates after cutoff_date and up to label_end_date.

label_df = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*) AS future_calendar_days_observed_30d,
        COUNT(*) FILTER (WHERE available = TRUE) AS future_available_days_30d,
        AVG(CASE WHEN available = TRUE THEN 1.0 ELSE 0.0 END) AS future_available_rate_30d,
        CASE
            WHEN AVG(CASE WHEN available = TRUE THEN 1.0 ELSE 0.0 END) <= :threshold
            THEN 1
            ELSE 0
        END AS high_demand_proxy
    FROM core.calendar_day
    WHERE date > CAST(:cutoff_date AS date)
      AND date <= CAST(:label_end_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "cutoff_date": cutoff_date,
        "label_end_date": label_end_date,
        "threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
    },
)

# TODO: convert numeric columns and make high_demand_proxy integer.

label_numeric_cols = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

for col in label_numeric_cols:
    label_df[col] = pd.to_numeric(label_df[col], errors="coerce")

label_df["high_demand_proxy"] = label_df["high_demand_proxy"].astype("Int64")

assert label_df["listing_id"].duplicated().sum() == 0

print(label_df.shape)
label_df.head()

(10480, 5)


,listing_id,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy
0,1443670960781261954,30,30,1.0,0
1,896043282611946316,30,0,0.0,1
2,39969190,30,0,0.0,1
3,958726726744532841,30,0,0.0,1
4,1272264495001498383,30,30,1.0,0


In [154]:
# Check label balance.

label_distribution = (
    label_df["high_demand_proxy"]
    .value_counts(dropna=False)
    .rename_axis("high_demand_proxy")
    .reset_index(name="count")
)

label_distribution["percentage"] = (
    label_distribution["count"] / label_distribution["count"].sum()
).round(4)

label_distribution

,high_demand_proxy,count,percentage
0,1,7994,0.7628
1,0,2486,0.2372


## 13. Join feature groups and label

Join all feature groups into one ML-ready table.

The final granularity must be:

```text
one row = one listing_id at one cutoff_date
```

Use an inner join with `label_df`, because rows without a target cannot be used for supervised learning.

In [155]:
# TODO: join static_features, review_features, calendar_features_all, and label_df.
feature_df = (
    static_features
    .merge(review_features, on="listing_id", how="left")
    .merge(calendar_features_all, on="listing_id", how="left")
    .merge(label_df, on="listing_id", how="inner")
)

# TODO: add cutoff_date and dataset_version columns.
# TODO: fill missing review count features with zero.
# TODO: handle missing days_since_last_review for listings with no reviews.

feature_df["cutoff_date"] = pd.to_datetime(cutoff_date)
feature_df["dataset_version"] = DATASET_VERSION

review_fill_zero_cols = [
    "total_reviews_before_cutoff",
    "unique_reviewers_before_cutoff",
    "avg_comment_len_before_cutoff",
    "max_comment_len_before_cutoff",
]

for col in review_fill_zero_cols:
    if col in feature_df.columns:
        feature_df[col] = feature_df[col].fillna(0)

if "days_since_last_review" in feature_df.columns:
    feature_df["days_since_last_review"] = feature_df["days_since_last_review"].fillna(9999)

assert feature_df["listing_id"].duplicated().sum() == 0

print(feature_df.shape)
feature_df.head()

(10480, 33)


,listing_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,...,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version
0,27886,Private room,Private room in houseboat,2,1.0,1.0,1.5,<NA>,3,356,...,0,0.000000,3.0,30.0,30,0,0.0,1,2026-08-11,v1_student
1,28871,Private room,Private room in rental unit,2,1.0,1.0,1.0,<NA>,2,730,...,14,0.466667,2.0,730.0,30,21,0.7,0,2026-08-11,v1_student
2,29051,Private room,Private room in condo,2,1.0,1.0,1.0,<NA>,2,730,...,16,0.533333,2.0,730.0,30,0,0.0,1,2026-08-11,v1_student
3,44391,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,<NA>,3,730,...,0,0.000000,3.0,730.0,30,0,0.0,1,2026-08-11,v1_student
4,48373,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,<NA>,3,1125,...,0,0.000000,3.0,1125.0,30,0,0.0,1,2026-08-11,v1_student


## 14. Drop unusable columns

Before saving, remove bad feature columns.

Drop columns that are:

- more than 95% missing,
- constant across all rows,

but protect target/audit columns.

In [156]:
protected_columns = {
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
}

HIGH_MISSING_DROP_THRESHOLD = 0.95

# TODO: compute missing rates.
# TODO: find high-missing columns outside protected columns.
# TODO: find constant columns outside protected columns.
# TODO: drop them from feature_df.

missing_rates = feature_df.isna().mean()

high_missing_cols = [
    col
    for col, rate in missing_rates.items()
    if rate > HIGH_MISSING_DROP_THRESHOLD and col not in protected_columns
]

constant_cols = [
    col
    for col in feature_df.columns
    if col not in protected_columns and feature_df[col].nunique(dropna=False) <= 1
]

columns_to_drop = sorted(set(high_missing_cols + constant_cols))

print("Columns to drop:", columns_to_drop)

feature_df = feature_df.drop(columns=columns_to_drop)

print("New shape:", feature_df.shape)

Columns to drop: ['listing_price']
New shape: (10480, 32)


## 15. Validate the final dataset

The validation step is mandatory.

Check:

1. no duplicate `listing_id + cutoff_date`,
2. target exists and is binary,
3. no missing target values,
4. no forbidden PII columns,
5. no future leakage columns in model inputs.

In [157]:
duplicate_count = feature_df.duplicated(subset=["listing_id", "cutoff_date"]).sum()
missing_target_count = feature_df["high_demand_proxy"].isna().sum()
unique_target_values = sorted(feature_df["high_demand_proxy"].dropna().unique().tolist())

forbidden_columns = {
    "host_id",
    "host_pseudo_id",
    "reviewer_id",
    "reviewer_pseudo_id",
    "review_id",
    "license",
    "bathrooms_text",
}

present_forbidden_columns = sorted(forbidden_columns.intersection(feature_df.columns))

label_only_columns = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

model_input_columns = [
    col for col in feature_df.columns
    if col not in label_only_columns
    and col not in ["listing_id", "cutoff_date", "dataset_version"]
]

future_leakage_columns = [
    col for col in model_input_columns
    if col.startswith("future_")
]

# TODO: add asserts for each validation rule.
# for example:
# assert duplicate_count == 0
assert duplicate_count == 0, f"Found duplicate listing_id + cutoff_date rows: {duplicate_count}"
assert missing_target_count == 0, f"Found missing target values: {missing_target_count}"
assert set(unique_target_values).issubset({0, 1}), f"Unexpected target values: {unique_target_values}"
assert len(present_forbidden_columns) == 0, f"Forbidden columns present: {present_forbidden_columns}"
assert len(future_leakage_columns) == 0, f"Future leakage columns found in model inputs: {future_leakage_columns}"

print("duplicate_count:", duplicate_count)
print("missing_target_count:", missing_target_count)
print("unique_target_values:", unique_target_values)
print("present_forbidden_columns:", present_forbidden_columns)
print("future_leakage_columns:", future_leakage_columns)
print("model_input_column_count:", len(model_input_columns))

duplicate_count: 0
missing_target_count: 0
unique_target_values: [0, 1]
present_forbidden_columns: []
future_leakage_columns: []
model_input_column_count: 25


In [158]:
missing_report = (
    feature_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_report.columns = ["column", "missing_rate"]

calendar_coverage_summary = (
    feature_df[["future_calendar_days_observed_30d"]]
    .describe()
    .reset_index()
)

display(missing_report.head(30))
display(label_distribution)
display(calendar_coverage_summary)

,column,missing_rate
0,beds,0.436641
1,bedrooms,0.029198
2,is_superhost,0.010973
3,bathrooms,0.001145
4,accommodates,0.000000
5,property_type,0.000000
6,room_type,0.000000
7,listing_id,0.000000
8,minimum_nights,0.000000
9,maximum_nights,0.000000


,high_demand_proxy,count,percentage
0,1,7994,0.7628
1,0,2486,0.2372


,index,future_calendar_days_observed_30d
0,count,10480.0
1,mean,30.0
2,std,0.0
3,min,30.0
4,25%,30.0
5,50%,30.0
6,75%,30.0
7,max,30.0


## 16. Save versioned outputs

Save:

- feature dataset,
- metadata,
- validation report,
- PII audit.

The MLflow notebook must read this output instead of querying raw database tables again.

In [159]:
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"
validation_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_validation_report.json"
pii_audit_path = FEATURE_DIR / f"pii_audit_{DATASET_VERSION}.csv"

feature_df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

try:
    feature_df.to_parquet(parquet_path, index=False)
    print("Saved Parquet:", parquet_path)
except ImportError:
    print("Parquet not saved because pyarrow/fastparquet is not installed.")
    print("Install pyarrow with: pip install pyarrow")

# TODO: build metadata dictionary.
metadata = {
    "dataset_version": DATASET_VERSION,
    "entity_column": ENTITY_COLUMN,
    "cutoff_date": str(cutoff_date),
    "history_start_date": str(history_start_date),
    "label_end_date": str(label_end_date),
    "past_window_days": PAST_WINDOW_DAYS,
    "future_window_days": FUTURE_WINDOW_DAYS,
    "high_demand_available_rate_threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
    "source_tables": [
        "core.listing",
        "core.host",
        "core.neighbourhood",
        "core.review",
        "core.calendar_day",
    ],
    "target_definition": (
        "high_demand_proxy = 1 if future_available_rate_30d <= "
        f"{HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD} else 0"
    ),
    "feature_rules": {
        "historical_features_use_data_on_or_before_cutoff": True,
        "target_uses_data_after_cutoff_only": True,
        "one_row_per_entity": True,
        "entity_key_in_final_dataset": "listing_id"
    },
    "pii_exclusion_rules": [
        "drop host_id from final dataset",
        "drop host_pseudo_id from final dataset",
        "drop review_id from final dataset",
        "drop reviewer_id from final dataset",
        "drop reviewer_pseudo_id from final dataset",
        "drop license from final dataset",
        "drop bathrooms_text from final dataset after parsing",
        "do not include raw text review content in final dataset",
    ],
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# TODO: build validation_report dictionary.
validation_report = {
    "duplicate_listing_cutoff_rows": int(duplicate_count),
    "missing_target_count": int(missing_target_count),
    "target_values": unique_target_values,
    "present_forbidden_columns": present_forbidden_columns,
    "future_leakage_columns_in_model_inputs": future_leakage_columns,
    "model_input_column_count": int(len(model_input_columns)),
    "missing_report": missing_report.to_dict(orient="records"),
    "label_distribution": label_distribution.to_dict(orient="records"),
    "calendar_coverage_summary": calendar_coverage_summary.to_dict(orient="records"),
}

with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)

pii_audit.to_csv(pii_audit_path, index=False)

print("Saved metadata:", metadata_path)
print("Saved validation report:", validation_path)
print("Saved PII audit:", pii_audit_path)

Saved CSV: c:\Users\aidah\Documents\MLOPS\HW_A\data\features\listing_availability_features_v1_student.csv
Saved Parquet: c:\Users\aidah\Documents\MLOPS\HW_A\data\features\listing_availability_features_v1_student.parquet
Saved metadata: c:\Users\aidah\Documents\MLOPS\HW_A\data\features\listing_availability_features_v1_student_metadata.json
Saved validation report: c:\Users\aidah\Documents\MLOPS\HW_A\data\features\listing_availability_features_v1_student_validation_report.json
Saved PII audit: c:\Users\aidah\Documents\MLOPS\HW_A\data\features\pii_audit_v1_student.csv


## 17. Final preview

Use this final cell to confirm the output shape and columns.

Before moving to Notebook 2, make sure:

- target column exists,
- model input columns do not include future columns,
- no forbidden PII columns are present,
- saved files exist in `data/features/`.

In [160]:
print("Final shape:", feature_df.shape)

display(feature_df.head())

feature_df.info()

Final shape: (10480, 32)


,listing_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms,minimum_nights,maximum_nights,instant_bookable,...,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version
0,27886,Private room,Private room in houseboat,2,1.0,1.0,1.5,3,356,False,...,0,0.000000,3.0,30.0,30,0,0.0,1,2026-08-11,v1_student
1,28871,Private room,Private room in rental unit,2,1.0,1.0,1.0,2,730,False,...,14,0.466667,2.0,730.0,30,21,0.7,0,2026-08-11,v1_student
2,29051,Private room,Private room in condo,2,1.0,1.0,1.0,2,730,False,...,16,0.533333,2.0,730.0,30,0,0.0,1,2026-08-11,v1_student
3,44391,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,3,730,False,...,0,0.000000,3.0,730.0,30,0,0.0,1,2026-08-11,v1_student
4,48373,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,3,1125,False,...,0,0.000000,3.0,1125.0,30,0,0.0,1,2026-08-11,v1_student


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10480 entries, 0 to 10479
Data columns (total 32 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   listing_id                            10480 non-null  int64         
 1   room_type                             10480 non-null  object        
 2   property_type                         10480 non-null  object        
 3   accommodates                          10480 non-null  int64         
 4   bedrooms                              10174 non-null  float64       
 5   beds                                  5904 non-null   float64       
 6   bathrooms                             10468 non-null  float64       
 7   minimum_nights                        10480 non-null  int64         
 8   maximum_nights                        10480 non-null  int64         
 9   instant_bookable                      10480 non-null  boolean       
 10